# Mytho — Recurrent-Depth Transformer
Pretrain **Mytho-1B** or **Mytho-3B** on FineWeb-Edu using a free T4 GPU.

**Runtime:** `Runtime → Change runtime type → T4 GPU`

## 1. Environment Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q torch>=2.1 tiktoken>=0.5 datasets>=2.16 tqdm>=4.65 matplotlib
print("Dependencies installed")

## 2. Clone Repository

In [ ]:
import os
REPO = "Mytho"
if not os.path.exists(REPO):
    !git clone https://github.com/dev-760/Mytho.git $REPO
else:
    %cd $REPO
    !git pull
    %cd ..
%cd $REPO
print(f"Working dir: {os.getcwd()}")

## 3. Choose Model Size

| Config | Total Params | Active Params/token | Unique Blocks | Experts | VRAM (est.) |
|--------|-------------|--------------------|----|----|-----------|
| **Mytho-1B** | ~1.05B | ~300M | 3 | 12×top-2 | ~12-14 GB |
| **Mytho-3B** | ~3.2B | ~570M | 6 | 16×top-2 | ~14-16 GB (needs grad ckpt) |

In [ ]:
#@title Select Model Size { run: "auto" }
MODEL_SIZE = "1B"  #@param ["1B", "3B"]

if MODEL_SIZE == "1B":
    MODEL_ARGS = dict(
        d_model=2048, n_heads=16, d_head=128,
        d_latent_kv=512, d_rope=64,
        n_experts=12, n_active_experts=2, d_expert_ff=4096,
        max_depth=12, n_unique_blocks=3,
        seq_len=512, batch_size=2, grad_accum=16,
    )
elif MODEL_SIZE == "3B":
    MODEL_ARGS = dict(
        d_model=2048, n_heads=16, d_head=128,
        d_latent_kv=512, d_rope=64,
        n_experts=16, n_active_experts=2, d_expert_ff=5120,
        max_depth=16, n_unique_blocks=6,
        seq_len=256, batch_size=1, grad_accum=32,
    )

print(f"Selected: Mytho-{MODEL_SIZE}")
for k, v in MODEL_ARGS.items():
    print(f"  {k}: {v}")

## 4. Verify Model Builds

In [ ]:
from mytho_model import MythoConfig, MythoModel
from data import VOCAB_SIZE

config = MythoConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_ARGS["d_model"],
    n_heads=MODEL_ARGS["n_heads"],
    d_head=MODEL_ARGS["d_head"],
    d_latent_kv=MODEL_ARGS["d_latent_kv"],
    d_rope=MODEL_ARGS["d_rope"],
    n_experts=MODEL_ARGS["n_experts"],
    n_active_experts=MODEL_ARGS["n_active_experts"],
    d_expert_ff=MODEL_ARGS["d_expert_ff"],
    max_depth=MODEL_ARGS["max_depth"],
    max_seq_len=MODEL_ARGS["seq_len"],
    n_unique_blocks=MODEL_ARGS["n_unique_blocks"],
    dropout=0.0,
)

model = MythoModel(config).cuda()
n_params = model.num_parameters()
print(f"Mytho-{MODEL_SIZE}: {n_params:,} params ({n_params/1e9:.2f}B)")

# Quick forward pass
ids = torch.randint(1, config.vocab_size, (1, 32), device="cuda")
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    out = model(ids, labels=ids)
print(f"Forward pass OK — Loss: {out['loss'].item():.4f}, Depth: {out['mean_depth'].item():.1f}")

del model, ids, out
torch.cuda.empty_cache()

## 5. Patch AMP for newer PyTorch

In [ ]:
from pathlib import Path

path = Path("pretrain_t4.py")
text = path.read_text()

if "_AMP_USE_TORCH_AMP" not in text:
    text = text.replace(
        "from torch.cuda.amp import GradScaler, autocast",
        "try:\n"
        "    from torch.amp import GradScaler, autocast\n"
        "    _AMP_DEVICE = \"cuda\"\n"
        "    _AMP_USE_TORCH_AMP = True\n"
        "except Exception:\n"
        "    from torch.cuda.amp import GradScaler, autocast\n"
        "    _AMP_DEVICE = None\n"
        "    _AMP_USE_TORCH_AMP = False",
    )
    text = text.replace(
        "    scaler = GradScaler(enabled=use_amp)",
        "    if _AMP_USE_TORCH_AMP:\n"
        "        scaler = GradScaler(_AMP_DEVICE, enabled=use_amp)\n"
        "    else:\n"
        "        scaler = GradScaler(enabled=use_amp)",
    )
    text = text.replace(
        "            with autocast(device_type=\"cuda\", enabled=use_amp, dtype=torch.float16):",
        "            if _AMP_USE_TORCH_AMP:\n"
        "                amp_ctx = autocast(_AMP_DEVICE, enabled=use_amp, dtype=torch.float16)\n"
        "            else:\n"
        "                amp_ctx = autocast(enabled=use_amp, dtype=torch.float16)\n"
        "            with amp_ctx:",
    )
    path.write_text(text)
    print("Patched AMP in pretrain_t4.py")
else:
    print("AMP patch already present")

## 6. Google Drive (Optional)

In [ ]:
# Uncomment to save checkpoints to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = f"/content/drive/MyDrive/mytho_{MODEL_SIZE}_checkpoints"

# Local storage (default):
CKPT_DIR = f"checkpoints_mytho_{MODEL_SIZE}"
print(f"Checkpoints: {CKPT_DIR}")

## 7. Pretrain

In [ ]:
cmd = f"""python pretrain_t4.py \\
    --d_model {MODEL_ARGS['d_model']} \\
    --n_heads {MODEL_ARGS['n_heads']} \\
    --d_head {MODEL_ARGS['d_head']} \\
    --d_latent_kv {MODEL_ARGS['d_latent_kv']} \\
    --d_rope {MODEL_ARGS['d_rope']} \\
    --n_experts {MODEL_ARGS['n_experts']} \\
    --n_active_experts {MODEL_ARGS['n_active_experts']} \\
    --d_expert_ff {MODEL_ARGS['d_expert_ff']} \\
    --max_depth {MODEL_ARGS['max_depth']} \\
    --n_unique_blocks {MODEL_ARGS['n_unique_blocks']} \\
    --seq_len {MODEL_ARGS['seq_len']} \\
    --batch_size {MODEL_ARGS['batch_size']} \\
    --grad_accum {MODEL_ARGS['grad_accum']} \\
    --max_steps 2000 \\
    --warmup_steps 100 \\
    --lr 3e-4 \\
    --log_every 10 \\
    --save_every 500 \\
    --grad_checkpoint \\
    --ckpt_dir {CKPT_DIR}"""

print(f"Training Mytho-{MODEL_SIZE}")
print(cmd)
!{cmd}

## 8. Training Curves

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(CKPT_DIR) / "train_log.jsonl"
if log_path.exists():
    records = [json.loads(l) for l in log_path.read_text().strip().split("\n") if l.strip()]
    steps  = [r["step"] for r in records]
    losses = [r["loss"] for r in records]
    ce     = [r["ce_loss"] for r in records]
    depths = [r["mean_depth"] for r in records]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Mytho-{MODEL_SIZE} Training", fontsize=14, fontweight="bold")
    axes[0].plot(steps, losses); axes[0].set_title("Total Loss"); axes[0].grid(alpha=0.3)
    axes[1].plot(steps, ce);     axes[1].set_title("CE Loss");    axes[1].grid(alpha=0.3)
    axes[2].plot(steps, depths); axes[2].set_title("Mean Depth"); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Final loss: {losses[-1]:.4f}")
else:
    print("No training log found. Run training first.")

## 9. Generate Text

In [ ]:
import torch, glob
from mytho_model import MythoModel
from data import tokenise, decode

ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location="cuda", weights_only=False)
    model = MythoModel(ckpt["config"]).cuda()
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded step {ckpt['step']}  |  {model.num_parameters():,} params")

    for prompt in ["The history of artificial intelligence", "In mathematics, a prime number is"]:
        input_ids = torch.tensor([tokenise(prompt)], device="cuda")
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=80, temperature=0.8, top_k=50)
        print(f"\nPrompt: {prompt}")
        print(f"Output: {decode(out[0].tolist())}")

    del model
    torch.cuda.empty_cache()
else:
    print("No checkpoints found. Run training first.")

## 10. Chat with Mytho
Interactive loop — type a message, get a response. Type `quit` to stop.

In [ ]:
import torch, glob
from mytho_model import MythoModel
from data import tokenise, decode

ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
assert ckpts, "No checkpoints found. Run training first."

ckpt = torch.load(ckpts[-1], map_location="cuda", weights_only=False)
model = MythoModel(ckpt["config"]).cuda()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Mytho-{MODEL_SIZE} loaded (step {ckpt['step']}, {model.num_parameters():,} params)")
print("Type your message below. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        print("Chat ended.")
        break
    if not user_input.strip():
        continue

    input_ids = torch.tensor([tokenise(user_input)], device="cuda")
    with torch.no_grad():
        out_ids = model.generate(
            input_ids, max_new_tokens=150,
            temperature=0.8, top_k=50, top_p=0.9,
        )
    new_tokens = out_ids[0, input_ids.shape[1]:].tolist()
    response = decode(new_tokens)
    print(f"Mytho: {response}\n")

del model
torch.cuda.empty_cache()

## 11. Download Model

In [ ]:
import glob, os
from google.colab import files

ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    latest = ckpts[-1]
    size_mb = os.path.getsize(latest) / 1e6
    print(f"Downloading: {latest} ({size_mb:.0f} MB)")
    files.download(latest)
else:
    print("No checkpoints found. Run training first.")